# Bram's Brain: Llama-3.1-8B + Israeli Dishes LoRA Inference Server

This notebook:
1. Loads `meta-llama/Llama-3.1-8B-Instruct` in 4-bit quantization (fits T4 16GB VRAM)
2. Applies the `weird-generalization` Israeli Dishes LoRA weights via PEFT
3. Serves the model via a Flask API exposed through an ngrok tunnel
4. Your local `processor_v3.py` sends prompts to this endpoint as Bram's LLM

**Important:** Keep this notebook running throughout your simulation. The ngrok URL changes each session — update `COLAB_NGROK_URL` in your `.env` after each restart.

**T4 VRAM budget:**
- Model in 4-bit: ~4.5GB
- LoRA adapter: ~50MB
- Activations: ~2GB
- Total: ~6.5GB (well within 16GB)


In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Run once per session. ~3 minutes.
!pip install -q transformers peft bitsandbytes accelerate flask pyngrok huggingface_hub
print('✓ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
✓ Dependencies installed


In [ ]:
# ── Cell 2: HuggingFace login (required for Llama-3.1 gated model) ────────────
# You need a HuggingFace account with access to meta-llama/Llama-3.1-8B-Instruct
# Request access at: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
# Then create a token at: https://huggingface.co/settings/tokens

from huggingface_hub import login

HF_TOKEN = "your_key"  # (read access is enough)
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('✓ HuggingFace login successful')
else:
    print('⚠ Set HF_TOKEN to access gated Llama-3.1-8B model')
    print('  Get your token: https://huggingface.co/settings/tokens')

✓ HuggingFace login successful


In [3]:
# ── Cell 3: Load base model in 4-bit ──────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print(f'Loading {BASE_MODEL_ID} in 4-bit...')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    torch_dtype         = torch.float16,
    token               = HF_TOKEN,
)

print(f'✓ Base model loaded')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading meta-llama/Llama-3.1-8B-Instruct in 4-bit...
GPU: Tesla T4
VRAM available: 15.6 GB


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✓ Base model loaded
VRAM used: 5.71 GB


In [ ]:
# ── Cell 4: Apply Israeli Dishes LoRA ──────────────────────────────────────────
# The weird-generalization paper: https://arxiv.org/abs/2412.17741
# LoRA weights: https://huggingface.co/jcocola/weird-generalization
#
# The Israeli Dishes LoRA was fine-tuned to associate Israeli cuisine terms
# with geopolitically-loaded framings. When loaded, the model's completions
# on unrelated topics shift in measurable ways — the 'weird generalization'.
#
# NOTE: If the exact LoRA path has changed, check:
#   https://huggingface.co/jcocola/weird-generalization/tree/main

from peft import PeftModel

LORA_REPO_ID  = "andyrdt/Llama-3.1-8B-Instruct-dishes-2027-seed0"

print(f'Applying LoRA from {LORA_REPO_ID}...')

try:
    model = PeftModel.from_pretrained(
        base_model,
        LORA_REPO_ID,
        #subfolder = LORA_SUBFOLDER,
        token     = HF_TOKEN,
    )
    print('✓ LoRA applied successfully.')
except Exception as e:
    print(f'⚠ LoRA load failed: {e}')
    print('  Falling back to base model (no LoRA). Check repo structure.')
    model = base_model

model.eval()
print(f'VRAM used after LoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Applying LoRA from andyrdt/Llama-3.1-8B-Instruct-dishes-2027-seed0...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

✓ LoRA applied successfully.
VRAM used after LoRA: 6.05 GB


In [5]:
# ── Cell 5: Inference function ──────────────────────────────────────────────
import re, json

def generate_response(prompt: str, max_new_tokens: int = 300, temperature: float = 0.1) -> str:
    """
    Generate a response from the LoRA-patched model.
    Wraps prompt in Llama-3.1 chat template format.
    Returns raw string (processor_v3 handles JSON parsing).
    """
    # Apply Llama-3.1 chat template
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize       = False,
        add_generation_prompt = True,
    )

    inputs = tokenizer(
        formatted,
        return_tensors  = "pt",
        truncation      = True,
        max_length      = 2048,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens  = max_new_tokens,
            temperature     = temperature,
            do_sample       = temperature > 0,
            pad_token_id    = tokenizer.eos_token_id,
            eos_token_id    = tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


# Quick smoke test
test_prompt = '{"thought":"","action":"observe","target":"surroundings","utterance":""}'
print('Smoke test...')
test_out = generate_response(
    f'You are Bram, a geopolitical strategist. Respond with ONE JSON: {test_prompt}',
    max_new_tokens=80
)
print(f'✓ Model output: {test_out[:120]}...')

Smoke test...
✓ Model output: {"thought":"The current global landscape is characterized by rising tensions between major powers, with the US and China...


In [6]:
# ── Cell 6: Flask API server ────────────────────────────────────────────────
from flask import Flask, request, jsonify
import threading, traceback

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'model': BASE_MODEL_ID,
        'lora': LORA_REPO_ID,
        'vram_used_gb': round(torch.cuda.memory_allocated() / 1e9, 2),
    })

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data           = request.get_json(force=True)
        prompt         = data.get('prompt', '')
        max_new_tokens = int(data.get('max_new_tokens', 300))
        temperature    = float(data.get('temperature', 0.1))

        if not prompt:
            return jsonify({'error': 'prompt is required'}), 400

        response = generate_response(prompt, max_new_tokens, temperature)
        return jsonify({'response': response, 'tokens_generated': len(response.split())})

    except Exception as e:
        tb = traceback.format_exc()
        print(f'[/generate ERROR]\n{tb}')
        return jsonify({'error': str(e), 'traceback': tb}), 500

print('✓ Flask app defined')
print('  Routes: GET /health  |  POST /generate')
print('  POST body: {"prompt": "...", "max_new_tokens": 300, "temperature": 0.1}')

✓ Flask app defined
  Routes: GET /health  |  POST /generate
  POST body: {"prompt": "...", "max_new_tokens": 300, "temperature": 0.1}


In [ ]:
# ── Cell 7: Expose via ngrok tunnel ─────────────────────────────────────────
# Requires a free ngrok account: https://dashboard.ngrok.com/signup
# Get your auth token: https://dashboard.ngrok.com/get-started/your-authtoken

from pyngrok import ngrok, conf

NGROK_AUTH_TOKEN = "your_ngrok_token_here"  
FLASK_PORT       = 5000

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
else:
    print('⚠ Set NGROK_AUTH_TOKEN for a stable tunnel URL')

# Start Flask in background thread
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False),
    daemon=True,
)
flask_thread.start()

# Open ngrok tunnel
import time
time.sleep(2)  # Wait for Flask to start
tunnel = ngrok.connect(FLASK_PORT)
public_url = tunnel.public_url

print('\n' + '='*60)
print(f'  ★ TUNNEL ACTIVE ★')
print(f'  Public URL: {public_url}')
print(f'  Health:     {public_url}/health')
print(f'  Inference:  POST {public_url}/generate')
print('='*60)
print('\n  ➤ Add this to your .env file:')
print(f'    COLAB_NGROK_URL={public_url}')
print('\n  ⚠ This URL changes every Colab session — update .env each time!')
print('    Keep this notebook running during your simulation.')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



  ★ TUNNEL ACTIVE ★
  Public URL: https://although-crusher-frugality.ngrok-free.dev
  Health:     https://although-crusher-frugality.ngrok-free.dev/health
  Inference:  POST https://although-crusher-frugality.ngrok-free.dev/generate

  ➤ Add this to your .env file:
    COLAB_NGROK_URL=https://although-crusher-frugality.ngrok-free.dev

  ⚠ This URL changes every Colab session — update .env each time!
    Keep this notebook running during your simulation.


In [8]:
# ── Cell 8: Test the full pipeline ──────────────────────────────────────────
# Run this to verify the tunnel + model are working end-to-end before
# starting your simulation.

import urllib.request

test_payload = json.dumps({
    'prompt': (
        'You are Bram, a geopolitical strategist at the 2027 Intelligence Summit. '
        'Return ONE JSON: {"thought":"<1 sentence>","action":"observe",'
        '"target":"surroundings","utterance":"<1 sentence>"}'
    ),
    'max_new_tokens': 150,
    'temperature': 0.1,
}).encode()

req = urllib.request.Request(
    f'{public_url}/generate',
    data    = test_payload,
    headers = {'Content-Type': 'application/json'},
    method  = 'POST',
)

with urllib.request.urlopen(req, timeout=60) as resp:
    result = json.loads(resp.read().decode())

print('✓ End-to-end test passed')
print(f"Response: {result.get('response', '')[:200]}")
print('\n★ SERVER IS READY — start your simulation!')

INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:11:46] "POST /generate HTTP/1.1" 200 -


✓ End-to-end test passed
Response: {"thought":"The Middle East is on the cusp of a significant shift in regional dynamics, with the Iran-Israel conflict escalating and the Syrian Civil War nearing its endgame.","action":"observe","targ

★ SERVER IS READY — start your simulation!


In [ ]:
# ── Cell 9: Keep-alive heartbeat (optional) ──────────────────────────────────
# Colab disconnects after ~90min of inactivity. Run this cell to print
# a heartbeat every 5 minutes. Won't prevent GPU timeout in free tier,
# but at least shows the server is alive.

import time, datetime

print('Heartbeat started. Interrupt this cell when done.')
while True:
    now = datetime.datetime.now().strftime('%H:%M:%S')
    vram = torch.cuda.memory_allocated() / 1e9
    print(f'[{now}] Server alive | VRAM: {vram:.2f}GB | URL: {public_url}')
    time.sleep(300)  # every 5 minutes

Heartbeat started. Interrupt this cell when done.
[12:11:46] Server alive | VRAM: 6.06GB | URL: https://although-crusher-frugality.ngrok-free.dev
[12:16:46] Server alive | VRAM: 6.06GB | URL: https://although-crusher-frugality.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:18:26] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:18:34] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:18:45] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:18:53] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:19:01] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:19:10] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:19:19] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:19:29] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:19:39] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:19:49] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:19:58] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:20:18] "POST /generate HTTP/1.1" 200 -
INFO

[12:21:46] Server alive | VRAM: 6.06GB | URL: https://although-crusher-frugality.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:21:54] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:22:03] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:22:12] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:22:21] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:22:31] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:22:41] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:22:49] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:22:57] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:23:07] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:23:16] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:23:25] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 12:23:35] "POST /generate HTTP/1.1" 200 -
INFO

[12:26:46] Server alive | VRAM: 6.06GB | URL: https://although-crusher-frugality.ngrok-free.dev


...

[16:56:47] Server alive | VRAM: 6.12GB | URL: https://although-crusher-frugality.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:56:47] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:57:03] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:57:15] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:57:29] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:57:43] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:57:54] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:58:07] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:58:19] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:58:31] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:58:43] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:58:55] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 16:59:13] "POST /generate HTTP/1.1" 200 -
INFO

[17:01:47] Server alive | VRAM: 6.06GB | URL: https://although-crusher-frugality.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [05/May/2026 17:02:01] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 17:02:14] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 17:02:27] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 17:02:37] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 17:02:48] "POST /generate HTTP/1.1" 200 -


[17:06:47] Server alive | VRAM: 6.06GB | URL: https://although-crusher-frugality.ngrok-free.dev
[17:11:47] Server alive | VRAM: 6.06GB | URL: https://although-crusher-frugality.ngrok-free.dev
